# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a practical guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, which reads datasets structured in accordance with the Croissant metadata schema.

The dataset contains detailed clinical and pathological data for 77 cancer survivors with second primary colorectal cancer, supporting exploratory and analytical studies in biomarker stratification, anatomical predilection, and MSI-H prevalence among cancer survivors.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This step fetches both machine-readable metadata and (optionally) records described by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Number of Authors: {len(metadata.author) if hasattr(metadata, 'author') else 0}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"Date Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")

# If available, print available record sets
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Available Record Sets @ids:")
    for rs in metadata.recordSet:
        print(f"- {rs['@id']}")
else:
    print("Warning: No record sets referenced in top-level metadata. Will attempt to discover via Croissant schema traversal.")

## 2. Data Overview
List all available record sets and their fields using their `@id`. All references to the dataset's entities use their Croissant `@id` for precision.

> Each record set in Croissant groups related rows (as in a data table), and each field corresponds to a column within that record set.

In [ ]:
# Discover available record sets in the Croissant schema
record_set_ids = []
fields_in_recordsets = dict()

# Croissant may expose recordSets on the metadata object itself, else introspect via dataset._jsonld
def is_record_set(x):
    return isinstance(x, dict) and x.get('@type') in ['cr:RecordSet', 'RecordSet', 'http://mlcommons.org/croissant/RecordSet']
def is_field(x):
    return isinstance(x, dict) and x.get('@type') in ['cr:Field', 'Field', 'http://mlcommons.org/croissant/Field']

# Flatten schema graph
def find_all_RecordSets(schema):
    if isinstance(schema, dict):
        if is_record_set(schema):
            yield schema
        for v in schema.values():
            yield from find_all_RecordSets(v)
    elif isinstance(schema, list):
        for v in schema:
            yield from find_all_RecordSets(v)

# Find all record sets
record_sets = list(find_all_RecordSets(dataset._jsonld))
# Get their @ids
record_set_ids = [rs['@id'] for rs in record_sets]

print(f"Discovered {len(record_set_ids)} record sets in the Croissant schema:")
for idx, rs in enumerate(record_sets):
    rs_id = rs['@id']
    print(f"\nRecord Set {idx+1} @id: {rs_id}")
    # Find fields in this record set
    field_objs = []
    if 'field' in rs:
        # Could be a single dict or list of dicts
        if isinstance(rs['field'], list):
            field_objs = rs['field']
        else:
            field_objs = [rs['field']]
        field_ids = []
        print("  Fields:")
        for f in field_objs:
            # field may be dict or @id as string
            if isinstance(f, dict):
                fid = f.get('@id', str(f))
                print(f"    - {fid}")
                field_ids.append(fid)
            elif isinstance(f, str):
                print(f"    - {f}")
                field_ids.append(f)
        fields_in_recordsets[rs_id] = field_ids
    else:
        print("  (No fields directly referenced in this record set)")
        fields_in_recordsets[rs_id] = []
if not record_set_ids:
    print("No record sets found. Refer to the Croissant documentation for fixing schema.")

## 3. Data Extraction
Extract data from each record set using the discovered `@id`. Data is loaded into pandas DataFrames for analysis. All fields/columns are referenced by their Croissant `@id`.

> **Note:** For this dataset, there is typically one main record set containing the patient/sample-level tabular data.

In [ ]:
# Prepare to extract records for every record set
# (If you know which record set contains the data table, you can select it. Here, we extract all tables.)
dataframes = {}
for rs_id in record_set_ids:
    try:
        print(f"\nLoading records from record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame shape: {df.shape}")
            print(f"Columns (field @ids): {list(df.columns)}\nSample records:")
            display(df.head(3))
        else:
            print("No records found for this record set.")
    except Exception as e:
        print(f"Error loading records for record set {rs_id}: {e}")

# Select the primary data table for EDA (use the first nonempty DataFrame)
main_recordset_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_recordset_id = rs_id
        break
if main_recordset_id is None:
    raise ValueError('No main data table found among record sets. Check dataset schema.')
else:
    print(f"\nMain data table selected for EDA: {main_recordset_id}")

## 4. Exploratory Data Analysis (EDA)
This section explores the main data table. We:
- Select a numeric field (e.g., patient's age or diagnosis interval, referenced by its `@id`).
- Filter records based on a value threshold.
- Normalize the field.
- Optionally group by a categorical field, also referenced by `@id`.

> **Note:** Replace field `@id`s below with those identified in your schema. For illustration, we try to select typical field names and fallback to the first numeric column found.

In [ ]:
# Access the main data table (DataFrame)
df = dataframes[main_recordset_id]

# Try to automatically select a numeric field by dtype or typical field names by @id
preferred_numeric_names = ["age", "cr:age", "dv:age", "interval_between_diagnoses", "dv:interval_between_diagnoses"]
numeric_field_id = None

# Search for a numeric column by @id or by dtype
for col in df.columns:
    if col.lower() in preferred_numeric_names:
        numeric_field_id = col
        break
# Fallback to any integer or float column
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    raise ValueError('No numeric field found for EDA!')
print(f"Using numeric field (by @id): {numeric_field_id}")

# Investigate value distribution
print(df[numeric_field_id].describe())

# Define a threshold (e.g., 10 or the 10th percentile)
if df[numeric_field_id].dtype.kind in 'biufc':
    threshold = np.percentile(df[numeric_field_id].dropna(), 10)
else:
    threshold = 10

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nRecords where {numeric_field_id} > {threshold:.2f} (showing top 5):")
display(filtered_df.head())

# Normalize this field (z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (first 5):")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a categorical field if available
# Try to pick a group field (e.g., 'sex', 'diagnosis', etc.)
preferred_group_names = ["sex", "cr:sex", "dv:sex", "anatomical_location", "dv:anatomical_location"]
group_field = None
for col in df.columns:
    if col.lower() in preferred_group_names:
        group_field = col
        break
# Fallback to any object column with few unique values
if group_field is None:
    for col in df.columns:
        if df[col].dtype == 'O' and df[col].nunique() <= 10:
            group_field = col
            break

if group_field:
    print(f"\nGrouping data by field: {group_field}")
    grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize numeric field distributions and relationships. Here, we render:
- Histogram and KDE of the selected numeric field
- Boxplot (if group field is present)
- Scatter or strip plot for more insights

In [ ]:
# Plot distributions with seaborn and matplotlib
plt.figure(figsize=(10, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

    plt.figure(figsize=(8,5))
    sns.stripplot(x=group_field, y=numeric_field_id, data=df, jitter=True)
    plt.title(f"{numeric_field_id} spread by {group_field}")
    plt.show()

## 6. Conclusion

This notebook showcased how to load and explore a clinical oncology tabular dataset using the Croissant schema and `mlcroissant` Python library:

- Dataset was loaded dynamically via schema URL.
- Record sets, fields, and columns were referenced and extracted using their Croissant `@id`.
- Common EDA and visualizations revealed structure and distributions in the data.

This workflow provides a foundation for further research, filtering, and modeling with FAIR clinical datasets.